# Lab 5 · Computer vision: semantic segmentation — Independent assignment

**BME · MSc Deep Learning / VITMMA19 · Fall 2026/2027**
Department of Telecommunications and Artificial Intelligence
**Instructor:** Dr. Mohammed Salah Al-Radhi

**60 minutes, including four minutes to check and submit · five tasks · 20 points**

### Challenge: label every pixel, and keep working when the lighting changes

A camera reads handwritten digits on forms. Instead of one label per image, the system must label **every pixel**: background, or which digit (0–9) the pixel belongs to. That is *semantic segmentation*, from today's lecture. The training images were recorded under good, even lighting; in use, the forms are often lit from one side and covered in pencil marks.

You will measure segmentation quality properly, turn a sliding-window classifier into a fully convolutional one, find out what skip connections are for, debug BatchNorm, build a residual block, and audit an AI-written augmentation before trusting it.

| Task | Minutes | Points |
| --- | ---: | ---: |
| T1 · Pixels, classes and the metric trap | 8 | 3 |
| T2 · From sliding window to fully convolutional | 10 | 4 |
| T3 · Where versus what: encoder–decoder and skip connections | 14 | 5 |
| T4 · BatchNorm and residual connections | 12 | 4 |
| T5 · Augmentation you can trust, then test once | 12 | 4 |
| Check and submit | 4 | — |

**How this assignment works.** Most TODOs are short, but each asks you to *decide* something first:

- **Prediction locks.** Before a task's training runs, record numerical predictions with `predict(...)`. The engine refuses to train until you have. The predictions are marked for being specific and reasoned, **not** for being right.
- **Plan boxes.** Before some code TODOs, write the plan in one or two lines (shapes, formula, or which check catches what). The plan earns marks; working code alone does not.
- **Checks, not answers.** The provided checks test properties of your code without revealing the solution. If a check fails, read its message and reason about why.

Use validation data for every decision; the test set opens once, in T5c. Each training run takes about 5–10 seconds on a CPU. Marks reward correct methods and explanations supported by **your own outputs**. Optional deeper investigations follow T5.

Submit **one executed `.ipynb` file to Moodle by the end of the lab**. Intentional TODOs stop the template until you complete them. Use the guided notebook, `01_BME_DL_Lab5_Guided.ipynb`, and the official documentation as references.

## Your details

**Name:** YOUR NAME
**Neptun code:** YOUR NEPTUN CODE
**Assistance declaration:** *None*, or name each tool (e.g. ChatGPT, Claude, Copilot), what you used it for, and how you checked its output. If you used AI, T5b asks you to test one of its suggestions.

## 0. Setup and rules — provided

Save a personal copy in Drive. **A CPU runtime is sufficient; a GPU is optional.** Run the setup cells first. No download is needed: the digits ship with scikit-learn.

**Data.** Each 24×24 scene contains **two digits**, 11–14 pixels tall. They usually touch, and in about two thirds of the scenes one digit partly covers the other (occlusion, lecture slide 39). Every pixel has a label: **0 = background**, **k + 1 = digit k**, so there are 11 classes. The digits come from the same stratified split as Labs 3–4, so training, validation and test scenes never share a digit image: **1,200 / 400 / 400 scenes**.

- **Training** images use *standard* conditions: mild uneven lighting, a few pencil strokes, slight noise.
- **Validation** has two views of the *same* 400 scenes: **standard**, and **harsh** (strong lighting from one side, more pencil strokes). The labels are identical, so the two views form a paired comparison.
- **Test** scenes are built only when you open them, once, in T5c.

**Fixed training budget (the engine enforces it):** 12 epochs, batch 32, Adam with a one-cycle learning-rate schedule (peak 1e-2), seed 42, pixel-wise cross-entropy, final-epoch model.

**Selection metric:** mean IoU (mIoU) on the **harsh** validation view, the deployment condition. Other metrics help you explain *why* a model wins.

In [ ]:
import copy
import math
import os
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from matplotlib.colors import ListedColormap
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix as sk_confusion_matrix
from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(2, os.cpu_count() or 1))
SEED = 42
SPLIT_SEED = 2026                    # the same stratified digit IDs as Labs 3-4
CANVAS, NC = 24, 11                  # 24x24 scenes; 11 classes: 0 = background, k+1 = digit k
GLYPH_SIZES = (11, 14)               # each digit is drawn 11-14 px tall
EPOCHS, BATCH_SIZE, MAX_LR = 12, 32, 1e-2   # fixed budget: Adam with a one-cycle learning-rate schedule
N_TRAIN, N_VAL, N_TEST = 1200, 400, 400     # scenes, two digits each
STANDARD = dict(amp=(0.0, 0.3), offset=(0.0, 0.1), strokes=(0, 2), noise=0.05)   # training conditions
HARSH = dict(amp=(0.5, 0.9), offset=(0.15, 0.35), strokes=(3, 6), noise=0.05)    # deployment conditions
TASKS = ['T3', 'T4', 'T5', 'guided', 'extension']
RUNS, PREDICTIONS, TEST_OPENED = {}, {}, False
pd.set_option('display.precision', 3)
pd.set_option('display.width', 170)

print('Python:', platform.python_version(), '| PyTorch:', torch.__version__,
      '| scikit-learn:', sklearn.__version__)
print('Device:', DEVICE, '| CPU threads:', torch.get_num_threads())
print(f'Budget per run: {EPOCHS} epochs | batch {BATCH_SIZE} | Adam one-cycle, max lr {MAX_LR} | seed {SEED}')

In [ ]:
#@title Provided data helpers — run this cell; expand to inspect
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_scenes(digits, pool, n, seed):
    """Clean 'ink' images and per-pixel labels. Two digits per scene, drawn only from the given pool of
    digit IDs, so training, validation and test scenes never share a digit image. Placement avoids overlap
    when there is room; on a 24x24 canvas the digits usually touch, and the second one often partly covers
    the first (occlusion): its strokes are drawn on top."""
    rng = np.random.default_rng(seed)
    ink = torch.zeros(n, 1, CANVAS, CANVAS)
    mask = torch.zeros(n, CANVAS, CANVAS, dtype=torch.long)
    for k in range(n):
        used = torch.zeros(CANVAS, CANVAS, dtype=torch.bool)
        for _ in range(2):
            i = pool[rng.integers(len(pool))]
            size = int(rng.integers(GLYPH_SIZES[0], GLYPH_SIZES[1] + 1))
            g = F.interpolate(torch.tensor(digits.images[i] / 16.0, dtype=torch.float32)[None, None],
                              size=(size, size), mode='bilinear', align_corners=False)[0, 0].clamp(0, 1)
            for _try in range(50):
                r, c = (int(v) for v in rng.integers(0, CANVAS - size + 1, 2))
                if not used[r:r + size, c:c + size].any():
                    break
            used[r:r + size, c:c + size] = True
            on = g > 0.35                                  # the annotation rule: stroke pixels
            ink[k, 0, r:r + size, c:c + size][on] = g[on]
            mask[k, r:r + size, c:c + size][on] = int(digits.target[i]) + 1
    return ink, mask

def make_stroke_bank(n=3000, seed=99):
    """Short random line segments used as background clutter. Clutter is labelled background."""
    rng = np.random.default_rng(seed)
    bank = torch.zeros(n, CANVAS, CANVAS)
    for k in range(n):
        r0, c0 = rng.uniform(0, CANVAS, 2)
        length, angle, value = rng.integers(4, 10), rng.uniform(0, math.pi), rng.uniform(0.3, 0.7)
        for t in range(length):
            rr, cc = int(round(r0 + t * math.sin(angle))), int(round(c0 + t * math.cos(angle)))
            if 0 <= rr < CANVAS and 0 <= cc < CANVAS:
                bank[k, rr, cc] = value
    return bank

STROKES = make_stroke_bank()
_YY, _XX = torch.meshgrid(torch.linspace(-1, 1, CANVAS), torch.linspace(-1, 1, CANVAS), indexing='ij')

def render(ink, gen, cond):
    """Photometric rendering of clean scenes: an illumination gradient in a random direction,
    a brightness offset, background clutter strokes and pixel noise. Labels never change:
    clutter is only drawn where there is no ink."""
    n = ink.shape[0]
    uniform = lambda lo, hi: lo + (hi - lo) * torch.rand(n, 1, 1, 1, generator=gen)
    theta = 2 * math.pi * torch.rand(n, 1, 1, 1, generator=gen)
    ramp = (torch.cos(theta) * _XX + torch.sin(theta) * _YY + 1) / 2        # 0 on one side, 1 on the other
    image = ink * (1 - uniform(*cond['amp']) * ramp) + uniform(*cond['offset']) * ramp
    lo, hi = cond['strokes']
    if hi > 0:
        count = torch.randint(lo, hi + 1, (n,), generator=gen)
        pick = torch.randint(0, len(STROKES), (n, hi), generator=gen)
        use = (torch.arange(hi)[None] < count[:, None]).float()[:, :, None, None]
        clutter = (STROKES[pick] * use).amax(1, keepdim=True)
        image = torch.where(ink < 0.05, torch.maximum(image, clutter), image)
    image = image + cond['noise'] * torch.randn(ink.shape, generator=gen)
    return image.clamp(0, 1)

def prepare_data():
    digits = load_digits()
    ids = np.arange(len(digits.target))
    dev_ids, test_ids = train_test_split(ids, test_size=0.2, stratify=digits.target, random_state=SPLIT_SEED)
    train_ids, val_ids = train_test_split(dev_ids, test_size=0.25, stratify=digits.target[dev_ids],
                                          random_state=SPLIT_SEED)
    assert set(train_ids).isdisjoint(val_ids) and set(train_ids).isdisjoint(test_ids)
    assert set(val_ids).isdisjoint(test_ids)
    tr_ink, tr_mask = make_scenes(digits, train_ids, N_TRAIN, seed=1)
    va_ink, va_mask = make_scenes(digits, val_ids, N_VAL, seed=2)
    tr_img = render(tr_ink, torch.Generator().manual_seed(11), STANDARD)
    data = dict(train_ink=tr_ink, train_img=tr_img, train_mask=tr_mask,
                val_ink=va_ink, val_mask=va_mask,
                val_standard=render(va_ink, torch.Generator().manual_seed(12), STANDARD),
                val_harsh=render(va_ink, torch.Generator().manual_seed(13), HARSH),
                mean=tr_img.mean().item(), std=tr_img.std().item())

    def final_test(run):
        """Teaching guard, not security: call once, after the decision is written down."""
        global TEST_OPENED
        if TEST_OPENED:
            raise RuntimeError('The test set was already opened. Reuse final_report; do not retune.')
        if run['task'] not in ('T3', 'T4', 'T5') or run['status'] != 'complete':
            raise ValueError('Choose a completed T3, T4 or T5 run.')
        TEST_OPENED = True
        te_ink, te_mask = make_scenes(digits, test_ids, N_TEST, seed=3)
        views = {'standard': render(te_ink, torch.Generator().manual_seed(14), STANDARD),
                 'harsh': render(te_ink, torch.Generator().manual_seed(15), HARSH)}
        report = {'run_name': run['name'], 'n_scenes': N_TEST}
        for view, x in views.items():
            pred = predict_masks(run['model'], x)
            report[view] = segmentation_scores(pred, te_mask)
            report[view + '_pred'] = pred
        report['images'], report['mask'] = views['harsh'], te_mask
        return report
    return data, final_test

MASK_COLOURS = ListedColormap(['#f2f2f2', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                               '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'])

def show_masks(rows, titles, n=6, suptitle=None):
    """rows: list of tensors, each (N, 1, H, W) images or (N, H, W) masks."""
    fig, axes = plt.subplots(len(rows), n, figsize=(1.7 * n, 1.75 * len(rows)), squeeze=False)
    for r, (data_row, title) in enumerate(zip(rows, titles)):
        for i in range(n):
            ax = axes[r, i]
            if data_row.dim() == 4:
                ax.imshow(data_row[i, 0], cmap='gray', vmin=0, vmax=1)
            else:
                ax.imshow(data_row[i], cmap=MASK_COLOURS, vmin=0, vmax=NC - 1, interpolation='nearest')
            ax.set_xticks([]); ax.set_yticks([])
        axes[r, 0].set_ylabel(title, fontsize=8)
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    plt.show()

In [ ]:
DATA, open_test_once = prepare_data()
print('Scenes  train / validation / test:', N_TRAIN, N_VAL, N_TEST, '(test scenes are built only when opened)')
print('Image tensor:', tuple(DATA['val_harsh'].shape), '| mask tensor:', tuple(DATA['val_mask'].shape),
      DATA['val_mask'].dtype)
print(f"Normalisation from standard training images only: mean {DATA['mean']:.4f}, std {DATA['std']:.4f}")
show_masks([DATA['val_standard'], DATA['val_harsh'], DATA['val_mask']],
           ['standard view', 'harsh view', 'label mask'],
           suptitle='The same validation scenes under two conditions, and their per-pixel labels')
print('Mask colours: grey = background, then digits 0-9 in order:',
      ', '.join(f'{k}' for k in range(10)))

### Supplied engine and plotting helpers

`run_experiment(name, model_fn, task, aug=None)` trains a fresh model under the fixed budget and records, for both validation views: **mIoU**, **foreground IoU** (is each pixel ink or background, ignoring which digit?), **digit accuracy** (among true digit pixels, the fraction labelled with the right digit) and **pixel accuracy**. Every run stays in the log; reusing a name with different settings raises an error.

`predict(task, expected)` records your predictions for a task. `prediction_check(task)` later puts them next to the observed values.

In [ ]:
#@title Provided engine: metrics, prediction locks, training, log
def normalise(x):
    return (x - DATA['mean']) / DATA['std']

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def predict_logits(model, images, batch_size=400):
    model.eval()
    return torch.cat([model(normalise(images[i:i + batch_size]).to(DEVICE)).cpu()
                      for i in range(0, len(images), batch_size)])

def predict_masks(model, images, batch_size=400):
    return predict_logits(model, images, batch_size).argmax(1)

def segmentation_scores(pred, mask):
    """Engine metrics (computed with scikit-learn). mIoU averages classes present in prediction or truth."""
    cm = sk_confusion_matrix(mask.flatten().numpy(), pred.flatten().numpy(), labels=np.arange(NC))
    union = cm.sum(0) + cm.sum(1) - np.diag(cm)
    per_class = np.where(union > 0, np.diag(cm) / np.maximum(union, 1), np.nan)
    fg_pred, fg_true = pred > 0, mask > 0
    return dict(miou=float(np.nanmean(per_class)),
                fg_iou=float((fg_pred & fg_true).sum() / max((fg_pred | fg_true).sum(), 1)),
                digit_acc=float((pred[fg_true] == mask[fg_true]).float().mean()),
                pixel_acc=float((pred == mask).float().mean()),
                per_class_iou=per_class)

def predict(task, expected):
    """Record predictions before running a task's experiments. expected = {run_name: {metric: value}}.
    Re-running the cell with identical values is fine; changing them after runs exist is not."""
    if task not in TASKS:
        raise ValueError(f'task must be one of {TASKS}')
    if not expected or any(v is None for d in expected.values() for v in d.values()):
        raise ValueError('Fill in every predicted value (numbers, not None).')
    if task in PREDICTIONS and PREDICTIONS[task]['expected'] != expected:
        if any(r['task'] == task for r in RUNS.values()):
            raise RuntimeError(f'{task} runs already exist, so its predictions are frozen. '
                               'Discuss a revised expectation in writing instead.')
    PREDICTIONS[task] = dict(expected=copy.deepcopy(expected), time=time.strftime('%H:%M:%S'))
    print(f'{task}: predictions recorded at {PREDICTIONS[task]["time"]} for', ', '.join(expected))

def run_experiment(name, model_fn, task, aug=None, p_aug=0.5, seed=SEED, verbose=True):
    """Fresh model; fixed budget; final-epoch model; logs standard and harsh validation scores.
    aug: None, or a condition dict like STANDARD/HARSH used to re-render a fraction p_aug of each
    training batch (photometric only, so masks stay valid)."""
    if task not in TASKS:
        raise ValueError(f'task must be one of {TASKS}')
    if task not in PREDICTIONS and task not in ('guided', 'extension'):
        raise RuntimeError(f'Record your {task} predictions with predict(...) before running {task} experiments.')
    if TEST_OPENED and task != 'extension':
        raise RuntimeError('The test set is open. Only task="extension" runs are allowed now.')
    set_seed(seed)
    model = model_fn().to(DEVICE)
    signature = (str(model), repr(aug), p_aug, seed, task, EPOCHS, BATCH_SIZE, MAX_LR)
    if name in RUNS:
        if RUNS[name]['signature'] != signature:
            raise ValueError(f'"{name}" already records a different experiment. Use a new name; '
                             'the earlier run stays in the log.')
        if verbose:
            print(f'{name}: settings unchanged, reusing the recorded run.')
        return RUNS[name]
    optimizer = torch.optim.Adam(model.parameters(), lr=MAX_LR)
    steps = EPOCHS * math.ceil(N_TRAIN / BATCH_SIZE)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=MAX_LR, total_steps=steps, pct_start=0.25)
    shuffle_gen = torch.Generator().manual_seed(seed + 1000)
    aug_gen = torch.Generator().manual_seed(seed + 2000)
    history, status, start = [], 'complete', time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = torch.randperm(N_TRAIN, generator=shuffle_gen)
        total = 0.0
        for s in range(0, N_TRAIN, BATCH_SIZE):
            idx = order[s:s + BATCH_SIZE]
            x = DATA['train_img'][idx]
            if aug is not None:
                fresh = render(DATA['train_ink'][idx], aug_gen, aug)
                pick = torch.rand(len(idx), 1, 1, 1, generator=aug_gen) < p_aug
                x = torch.where(pick, fresh, x)
            loss = F.cross_entropy(model(normalise(x).to(DEVICE)), DATA['train_mask'][idx].to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
            total += loss.item() * len(idx)
        if not math.isfinite(total):
            status = 'failed'
            break
        history.append(dict(epoch=epoch, train_loss=total / N_TRAIN))
    run = dict(name=name, task=task, model=model.eval(), model_fn=model_fn.__name__, aug=aug, p_aug=p_aug,
               seed=seed, status=status, signature=signature, history=pd.DataFrame(history),
               parameters=count_parameters(model), seconds=time.perf_counter() - start)
    if status == 'complete':
        for view in ('standard', 'harsh'):
            scores = segmentation_scores(predict_masks(model, DATA['val_' + view]), DATA['val_mask'])
            run.update({f'{view}_{k}': v for k, v in scores.items()})
        run['train_loss'] = history[-1]['train_loss']
    RUNS[name] = run
    if verbose:
        print(f"{name:20s} {status} | params {run['parameters']:7,d} | {run['seconds']:5.1f} s | "
              f"standard mIoU {run.get('standard_miou', float('nan')):.3f} | "
              f"harsh mIoU {run.get('harsh_miou', float('nan')):.3f}")
    return run

LOG_COLUMNS = ['name', 'task', 'model_fn', 'parameters', 'seconds', 'train_loss',
               'standard_miou', 'standard_fg_iou', 'standard_digit_acc', 'standard_pixel_acc',
               'harsh_miou', 'harsh_fg_iou', 'harsh_digit_acc']

def experiment_table(tasks=None):
    rows = [{k: r.get(k) for k in LOG_COLUMNS} for r in RUNS.values() if tasks is None or r['task'] in tasks]
    return pd.DataFrame(rows, columns=LOG_COLUMNS)

def prediction_check(task):
    """Your recorded predictions next to the observed validation values."""
    rows = []
    for run_name, metrics in PREDICTIONS[task]['expected'].items():
        for metric, guess in metrics.items():
            observed = RUNS[run_name].get(metric) if run_name in RUNS else None
            rows.append(dict(run=run_name, metric=metric, predicted=guess, observed=observed,
                             error=None if observed is None else observed - guess))
    return pd.DataFrame(rows)

In [ ]:
#@title Provided analysis and plotting helpers
def show_predictions(runs, view='harsh', n=6, start=0):
    images = DATA['val_' + view][start:start + n]
    rows, titles = [images, DATA['val_mask'][start:start + n]], [f'{view} image', 'true mask']
    for run in runs:
        rows.append(predict_masks(run['model'], images))
        titles.append(run['name'])
    show_masks(rows, titles, n=n, suptitle=f'Validation scenes ({view} view)')

def plot_losses(runs, title):
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for run in runs:
        h = run['history']
        if len(h):
            ax.plot(h.epoch, h.train_loss, marker='o', ms=3, label=run['name'])
    ax.set(xlabel='Epoch', ylabel='Training cross-entropy', yscale='log', title=title)
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

def per_class_table(runs, view='harsh'):
    names = ['background'] + [f'digit {k}' for k in range(10)]
    return pd.DataFrame({r['name']: r[f'{view}_per_class_iou'] for r in runs}, index=names)

def measure_receptive_field(layers, size=64, seed=0):
    """Empirical receptive field: input pixels whose gradient reaches the central output unit."""
    torch.manual_seed(seed)
    net = nn.Sequential(*layers).eval()
    x = torch.randn(4, 1, size, size, requires_grad=True)
    out = net(x)
    out[:, :, out.shape[-2] // 2, out.shape[-1] // 2].sum().backward()
    nz = torch.nonzero(x.grad.abs().sum((0, 1)))
    return int(nz[:, 0].max() - nz[:, 0].min() + 1)

def time_call(fn, repeats=3):
    best = float('inf')
    for _ in range(repeats):
        start = time.perf_counter()
        out = fn()
        best = min(best, time.perf_counter() - start)
    return out, best

## T1 · Pixels, classes and the metric trap · 3 points · 8 minutes

**a.** Before computing anything, predict three numbers for the validation masks: the fraction of pixels that are background, and the **pixel accuracy** and **mIoU** of a "model" that labels every pixel as background. Then run the cell. **(1 point)**

**b.** Implement the confusion matrix for masks with a single `torch.bincount`, and per-class IoU from it. First compute the small example below **by hand**, write your values into `HAND_IOU`, and only then run the checks. **(1 point)**

**c.** Two broken predictions are provided. **A** labels every digit correctly but makes each digit one pixel too thick. **B** has perfect outlines but labels every 3 as an 8. Predict which has the higher pixel accuracy and which the higher mIoU, then compute them with *your* functions. Which would you rather deploy in a digit reader, and what does mIoU reward? **(1 point)**

In [ ]:
# TODO T1a: your predictions BEFORE running this cell (numbers between 0 and 1).
T1A_PREDICTION = dict(background_fraction=None, all_background_pixel_acc=None, all_background_miou=None)

assert all(isinstance(v, (int, float)) for v in T1A_PREDICTION.values()), 'Write your three predictions first.'
mask = DATA['val_mask']
baseline = segmentation_scores(torch.zeros_like(mask), mask)
observed = dict(background_fraction=(mask == 0).float().mean().item(),
                all_background_pixel_acc=baseline['pixel_acc'], all_background_miou=baseline['miou'])
display(pd.DataFrame({'predicted': T1A_PREDICTION, 'observed': observed}).round(3))
print('Validation pixels per class (background, digit 0, ..., digit 9):', torch.bincount(mask.flatten(), minlength=NC).tolist())

### T1a · Your answer

**Predicted vs observed, and why the all-background mIoU is so much lower than its pixel accuracy:** YOUR ANSWER (2 sentences)

### T1b · A worked example by hand first

A 3×4 image with four classes (0, 1, 2, 3). Rows are image rows.

```
truth        prediction
0 0 1 1      0 1 1 1
0 2 2 1      0 2 0 1
0 0 2 1      0 0 2 2
```

For each class, IoU = |prediction ∩ truth| / |prediction ∪ truth|. Class 3 appears in neither mask.

### T1b · Plan (write before coding)

**For class *k*, which cells of the confusion matrix (rows = truth, columns = prediction) are TP, FP and FN? Write IoU_k using the diagonal, row sum and column sum:** YOUR ANSWER
**Why must a class absent from both masks be NaN rather than 0 or 1 when averaging?** YOUR ANSWER

In [ ]:
# TODO T1b (1): your hand-computed IoU for classes 0, 1, 2, 3 (use float('nan') for an undefined class).
HAND_IOU = [None, None, None, None]

def pixel_confusion(pred, mask, n_classes=NC):
    """Confusion matrix of two integer masks (any shape): rows = true class, columns = predicted class.
    Use ONE torch.bincount call on a combined index; no Python loop over pixels or classes."""
    # TODO T1b (2)
    raise NotImplementedError('T1b: implement pixel_confusion.')

def iou_from_confusion(cm):
    """Per-class IoU as a float tensor; NaN for a class absent from both prediction and truth."""
    # TODO T1b (3)
    raise NotImplementedError('T1b: implement iou_from_confusion.')

In [ ]:
#@title Checks for T1b (run after completing the cell above)
assert None not in HAND_IOU, 'Compute HAND_IOU by hand first.'
truth = torch.tensor([[0, 0, 1, 1], [0, 2, 2, 1], [0, 0, 2, 1]])
guess = torch.tensor([[0, 1, 1, 1], [0, 2, 0, 1], [0, 0, 2, 2]])
cm_small = pixel_confusion(guess, truth, n_classes=4)
assert cm_small.shape == (4, 4) and int(cm_small.sum()) == 12, 'The matrix must be 4x4 and count all 12 pixels.'
mine = iou_from_confusion(cm_small)
hand = torch.tensor(HAND_IOU, dtype=torch.float64)
same = torch.isclose(mine.double(), hand, atol=0.01) | (torch.isnan(mine.double()) & torch.isnan(hand))
print('Your function:', [round(v, 3) for v in mine.tolist()], '| your hand values:', [round(v, 3) for v in HAND_IOU])
assert bool(same.all()), 'Your function and your hand calculation disagree: recheck one of them.'
# A larger random check against the engine's scikit-learn implementation.
g = torch.Generator().manual_seed(0)
a, b = torch.randint(0, NC, (50, 24, 24), generator=g), torch.randint(0, NC, (50, 24, 24), generator=g)
b[b == 7] = 0                                       # class 7 never predicted
a[a == 9], b[b == 9] = 1, 1                         # class 9 absent from both
ref = segmentation_scores(b, a)['per_class_iou']
got = iou_from_confusion(pixel_confusion(b, a)).numpy()
assert np.allclose(np.nan_to_num(got, nan=-1), np.nan_to_num(ref, nan=-1), atol=1e-6), 'Mismatch on random masks.'
assert np.isnan(got[9]) and got[7] == 0, 'An absent class should be NaN; a missed class should be 0.'
print('T1b checks passed. mIoU of the small example:', round(float(np.nanmean(mine.numpy())), 3))

### T1b · Result

**Your hand values and the small example's mIoU; one thing the random check tests that the small example does not:** YOUR ANSWER

In [ ]:
#@title Provided: two broken predictions of the validation masks
mask = DATA['val_mask']
one_hot = F.one_hot(mask, NC).permute(0, 3, 1, 2).float()
one_hot[:, 0] = 0
grown = F.max_pool2d(one_hot, 3, stride=1, padding=1)               # each digit grows by one pixel
PRED_A = torch.where((mask == 0) & (grown.sum(1) > 0), grown.argmax(1), mask)
PRED_B = torch.where(mask == 4, torch.full_like(mask, 9), mask)     # every 3 (label 4) becomes 8 (label 9)
show_masks([DATA['val_standard'], mask, PRED_A, PRED_B], ['image', 'truth', 'A: thick', 'B: 3 -> 8'], n=6)

In [ ]:
# TODO T1c: predict BEFORE running: 'A' or 'B'.
T1C_PREDICTION = dict(higher_pixel_acc=None, higher_miou=None)

assert all(v in ('A', 'B') for v in T1C_PREDICTION.values()), "Write 'A' or 'B' for both predictions."
rows = {}
for label, pred in [('A: thick', PRED_A), ('B: 3 -> 8', PRED_B)]:
    per_class = iou_from_confusion(pixel_confusion(pred, mask))
    rows[label] = dict(pixel_acc=(pred == mask).float().mean().item(),
                       miou=float(np.nanmean(per_class.numpy())),
                       iou_background=per_class[0].item(), iou_digit3=per_class[4].item(),
                       iou_digit8=per_class[9].item(), changed_pixels=int((pred != mask).sum()))
display(pd.DataFrame(rows).round(3))

### T1c · Your answer

**Were your predictions right? Explain the mIoU of A using the shape of digit strokes:** YOUR ANSWER (2 sentences)
**Which prediction would you deploy in a digit reader, and what does mIoU reward that a digit reader may not need?** YOUR ANSWER (2 sentences)

## T2 · From sliding window to fully convolutional · 4 points · 10 minutes

A teammate built the lecture's first idea (slide 67): a **patch classifier** that looks at a 15×15 patch and predicts the label of its centre pixel. Segmenting an image means sliding it over every pixel. The provided cell trains it on random patches and times the sliding window.

**a.** Before running: how many patch forward passes does one 24×24 image need, and roughly how many times faster do you expect a fully convolutional version to be? **(1 point)**

**b.** Convert the patch classifier into an **equivalent fully convolutional network** that scores every pixel in one pass (slide 68). The convolutional layers already slide; only the final `nn.Linear` must become a convolution with copied weights. Write the plan first. The check compares your network with the sliding window on real images. **(2 points)**

**c.** Evaluate the converted network on both validation views. Use its receptive field and the digit size to explain why it finds *where* the ink is far better than *which* digit it is. **(1 point)**

In [ ]:
#@title The teammate's patch classifier (provided; read it)
PATCH = 15
PAD = PATCH // 2

class PatchClassifier(nn.Module):
    """Classifies the centre pixel of a 1 x 15 x 15 patch."""
    def __init__(self, width=8):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, width, 3), nn.ReLU(),        # no padding inside a patch:
                                      nn.Conv2d(width, width, 3), nn.ReLU(),    # 15 -> 13 -> 11 -> 9
                                      nn.Conv2d(width, width, 3), nn.ReLU())
        self.classifier = nn.Linear(width * 9 * 9, NC)

    def forward(self, patches):
        return self.classifier(self.features(patches).flatten(1))

def pad_scene(images):
    """Normalise, then pad by 7 with the value of an empty pixel so every pixel has a full 15x15 patch."""
    return F.pad(normalise(images), (PAD, PAD, PAD, PAD), value=-DATA['mean'] / DATA['std'])

def sliding_window_logits(net, images):
    """Slide the patch classifier over every pixel: one forward pass per pixel (batched)."""
    padded = pad_scene(images)
    patches = F.unfold(padded, PATCH).transpose(1, 2).reshape(-1, 1, PATCH, PATCH)   # (N*H*W, 1, 15, 15)
    with torch.no_grad():
        net.eval()
        scores = torch.cat([net(patches[i:i + 8192]) for i in range(0, len(patches), 8192)])
    return scores.view(len(images), CANVAS, CANVAS, NC).permute(0, 3, 1, 2)

def train_patch_classifier(n_patches=20000, epochs=4, seed=SEED):
    set_seed(seed)
    net = PatchClassifier()
    optimizer = torch.optim.Adam(net.parameters(), lr=3e-3)
    gen = torch.Generator().manual_seed(seed)
    padded = pad_scene(DATA['train_img'])
    start = time.perf_counter()
    for _ in range(epochs):
        img = torch.randint(0, N_TRAIN, (n_patches,), generator=gen)
        r, c = torch.randint(0, CANVAS, (n_patches,), generator=gen), torch.randint(0, CANVAS, (n_patches,), generator=gen)
        rows = (r[:, None] + torch.arange(PATCH))[:, :, None]
        cols = (c[:, None] + torch.arange(PATCH))[:, None, :]
        patches = padded[img[:, None, None], 0, rows, cols][:, None]
        labels = DATA['train_mask'][img, r, c]
        net.train()
        for s in range(0, n_patches, 256):
            loss = F.cross_entropy(net(patches[s:s + 256]), labels[s:s + 256])
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f'Patch classifier trained on {epochs} x {n_patches:,} random patches in {time.perf_counter() - start:.1f} s')
    return net.eval()

In [ ]:
# TODO T2a: predict BEFORE running.
T2A_PREDICTION = dict(patch_passes_per_image=None, expected_speedup=None)

assert all(isinstance(v, (int, float)) for v in T2A_PREDICTION.values()), 'Write both predictions first.'
patch_net = train_patch_classifier()
SAMPLE = DATA['val_standard'][:50]
sw_logits, sw_seconds = time_call(lambda: sliding_window_logits(patch_net, SAMPLE), repeats=1)
print(f'Sliding window on {len(SAMPLE)} images: {len(SAMPLE) * CANVAS * CANVAS:,} patch passes, {sw_seconds:.2f} s')

### T2b · Plan (write before coding)

**Shape of `patch_net.classifier.weight`, and the shape your convolution weight must have:** YOUR ANSWER
**Why a plain reshape (`.view`/`.reshape`) is correct here, and why `.t()` or `.permute(...)` would silently give wrong scores:** YOUR ANSWER

In [ ]:
class PaddedFCN(nn.Module):
    """Pads the scene once, then applies a fully convolutional network to it."""
    def __init__(self, body):
        super().__init__()
        self.body = body

    def forward(self, x):          # x is already normalised by the engine/predict helpers
        return self.body(F.pad(x, (PAD, PAD, PAD, PAD), value=-DATA['mean'] / DATA['std']))

def convolutionalise(net):
    """Return a fully convolutional network that gives the same scores as sliding `net` over every pixel."""
    linear = net.classifier
    conv = None   # TODO T2b: an nn.Conv2d that applies `linear` at every position of the feature map
    # TODO T2b: copy linear's weight (reshaped) and bias into conv, without tracking gradients
    if conv is None:
        raise NotImplementedError('T2b: build the convolution.')
    return PaddedFCN(nn.Sequential(net.features, conv)).eval()

In [ ]:
#@title Check for T2b: equal scores, and the speed-up
fcn_from_patches = convolutionalise(patch_net)
fc_logits, fc_seconds = time_call(lambda: predict_logits(fcn_from_patches, SAMPLE))
assert fc_logits.shape == sw_logits.shape == (len(SAMPLE), NC, CANVAS, CANVAS), 'Output must be N x 11 x 24 x 24.'
gap = (fc_logits - sw_logits).abs().max().item()
print(f'Largest score difference from the sliding window: {gap:.2e}')
assert gap < 1e-3, 'Scores differ: check the weight shape, the reshape order, and the bias.'
print(f'Fully convolutional: {fc_seconds * 1000:.1f} ms vs sliding window {sw_seconds * 1000:.0f} ms '
      f'-> {sw_seconds / fc_seconds:.0f}x faster for identical scores.')

In [ ]:
# T2c: evaluate the converted network (no TODO code; answer below using these numbers).
rf = measure_receptive_field(list(patch_net.features) + [convolutionalise(patch_net).body[1]])
for view in ('standard', 'harsh'):
    s = segmentation_scores(predict_masks(fcn_from_patches, DATA['val_' + view]), DATA['val_mask'])
    print(f"{view:8s} mIoU {s['miou']:.3f} | foreground IoU {s['fg_iou']:.3f} | digit accuracy {s['digit_acc']:.3f}")
print('Receptive field of one output pixel:', rf, 'px | digits are', GLYPH_SIZES, 'px tall')
show_masks([DATA['val_standard'][:6], DATA['val_mask'][:6], predict_masks(fcn_from_patches, DATA['val_standard'][:6])],
           ['image', 'truth', 'patch FCN'], n=6)

### T2 · Your answers

**T2a: your predictions vs the measured speed-up; where does the sliding window waste its time?** YOUR ANSWER
**T2b: why are the two networks' scores identical (not just similar)?** YOUR ANSWER
**T2c: foreground IoU vs digit accuracy, explained with the receptive field and digit size; what would a pixel near the edge of a 14-pixel digit need to see?** YOUR ANSWER (3 sentences)

## T3 · Where versus what: encoder–decoder and skip connections · 5 points · 14 minutes

Slide 69: downsample to gain context (*what*), upsample to recover resolution (*where*). You will compare three fully convolutional designs trained under the same budget:

| Model | Idea |
| --- | --- |
| `fcn_shallow` (provided) | three 3×3 convolutions and a 9×9 output convolution at full resolution: the T2 receptive field (15 px), no downsampling |
| `encdec_noskip` | `TinyUNet(skip=False)`: 24 → 12 → 6 → 12 → 24; the decoder sees only the 6×6 bottleneck |
| `unet` | `TinyUNet(skip=True)`: the same, plus skip connections that concatenate encoder features into the decoder (slide 52: skip = add or **concatenate**) |

**a.** Complete `TinyUNet`, then record predictions for **foreground IoU** and **digit accuracy** (standard view) of all three models. **(1 point)**

**b.** Train the three models and compare predicted with observed values. **(2 points)**

**c.** Explain the results as *where* versus *what*, using the numbers, the pictures and the parameter counts. Is the winner explained by size? Which failure on the **harsh** view does no architecture here fix? **(2 points)**

In [ ]:
#@title Provided building blocks and the shallow FCN
def conv_bn_relu(c_in, c_out):
    return [nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU()]

def double_conv(c_in, c_out):
    return nn.Sequential(*conv_bn_relu(c_in, c_out), *conv_bn_relu(c_out, c_out))

def fcn_shallow():
    w = 8
    return nn.Sequential(nn.Conv2d(1, w, 3, padding=1), nn.ReLU(), nn.Conv2d(w, w, 3, padding=1), nn.ReLU(),
                         nn.Conv2d(w, w, 3, padding=1), nn.ReLU(), nn.Conv2d(w, NC, 9, padding=4))

### T3a · Plan (write before coding)

**Channels entering `dec2` and `dec1`, with and without skips (w = 8):** YOUR ANSWER
**Why must the upsampled tensor match the skip tensor's height and width, and which dimension does `torch.cat` join?** YOUR ANSWER

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, skip=True, w=8):
        super().__init__()
        self.skip = skip
        self.enc1 = double_conv(1, w)               # 24 x 24, w channels
        self.enc2 = double_conv(w, 2 * w)           # 12 x 12, 2w channels
        self.bottleneck = double_conv(2 * w, 4 * w) # 6 x 6, 4w channels
        dec2_in = None   # TODO T3a: channels entering dec2 (depends on self.skip)
        dec1_in = None   # TODO T3a: channels entering dec1 (depends on self.skip)
        if dec2_in is None or dec1_in is None:
            raise NotImplementedError('T3a: set dec2_in and dec1_in.')
        self.dec2 = nn.Sequential(*conv_bn_relu(dec2_in, 2 * w))
        self.dec1 = nn.Sequential(*conv_bn_relu(dec1_in, w))
        self.head = nn.Conv2d(w, NC, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        b = self.bottleneck(F.max_pool2d(e2, 2))
        # TODO T3a: upsample b to e2's size (F.interpolate, mode='bilinear', align_corners=False),
        #           concatenate e2 if self.skip, apply dec2. Then the same one level up with e1 and dec1.
        raise NotImplementedError('T3a: write the decoder.')
        return self.head(d1)

def encdec_noskip():
    return TinyUNet(skip=False)

def unet():
    return TinyUNet(skip=True)

In [ ]:
#@title Checks for T3a
for fn in (fcn_shallow, encdec_noskip, unet):
    out = fn()(torch.zeros(2, 1, CANVAS, CANVAS))
    assert out.shape == (2, NC, CANVAS, CANVAS), f'{fn.__name__} must return 2 x 11 x 24 x 24 scores.'
    print(f'{fn.__name__:14s} {count_parameters(fn()):7,d} parameters')
assert count_parameters(unet()) > count_parameters(encdec_noskip()), 'Skips should add input channels to the decoder.'
probe = unet().eval()
x = torch.randn(1, 1, CANVAS, CANVAS)
with torch.no_grad():
    base = probe(x)
    probe.enc1[-1].register_forward_hook(lambda m, i, o: o * 0)      # silence the e1 skip path
    assert not torch.allclose(base, probe(x)), 'With skip=True, the output must depend on e1 through the skip.'
print('T3a checks passed.')

In [ ]:
# TODO T3a: record your predictions (fractions between 0 and 1) BEFORE the T3 runs.
predict('T3', {
    'T3_fcn_shallow':   dict(standard_fg_iou=None, standard_digit_acc=None),
    'T3_encdec_noskip': dict(standard_fg_iou=None, standard_digit_acc=None),
    'T3_unet':          dict(standard_fg_iou=None, standard_digit_acc=None),
})

In [ ]:
t3_runs = [run_experiment(f'T3_{fn.__name__}', fn, 'T3') for fn in (fcn_shallow, encdec_noskip, unet)]
display(experiment_table(['T3']).drop(columns=['task']))
display(prediction_check('T3').round(3))
show_predictions(t3_runs, view='standard')
show_predictions(t3_runs, view='harsh')

### T3b–c · Your interpretation

**Observed foreground IoU and digit accuracy of the three models; which predictions were furthest off, and why?** YOUR VALUES AND ANSWER
**Where vs what: what does each design get right, and what does the skip connection add? Is size the explanation?** YOUR ANSWER (3–4 sentences)
**The harsh view: which metric drops most for the best model, and what in the images causes it?** YOUR ANSWER (2 sentences)

## T4 · BatchNorm and residual connections · 4 points · 12 minutes

**a.** A teammate evaluated your T3 U-Net with their own function and reports that the score *depends on the evaluation batch size*, and that the model "got worse" after they evaluated it. Design a small experiment that finds the cause: for `mode` in train/eval and batch sizes 1, 8 and 400, restore the trained weights, predict the standard view, score it, and check whether any BatchNorm running statistic changed. Explain the cause, then the strange extra finding below. **(1 point)**

**b.** Implement the residual block of slide 52 (conv–BN–ReLU–conv–BN, add the shortcut, then ReLU; a 1×1 conv + BN shortcut when channel counts differ). Before running the gradient diagnostic, predict how the first-layer gradient of a **32-layer plain network** compares with its last layer. **(2 points)**

**c.** Train three 16-layer networks, **plain**, **plain + BatchNorm** and **residual**, under the same budget. Predict their standard-view mIoU first, then explain the results. Connect the worst one to a number you computed in T1a. **(1 point)**

In [ ]:
#@title The teammate's evaluation (provided; read it)
def teammate_evaluate(model, images, mask, batch_size):
    with torch.no_grad():
        pred = torch.cat([model(normalise(images[i:i + batch_size])).argmax(1)
                          for i in range(0, len(images), batch_size)])
    return segmentation_scores(pred, mask)['miou']

def bn_stats_changed(model, reference_state):
    """True if any BatchNorm running mean/variance differs from the reference state dict."""
    return any(not torch.equal(v, reference_state[k]) for k, v in model.state_dict().items()
               if 'running_' in k)

net = RUNS['T3_unet']['model']
CLEAN_STATE = copy.deepcopy(net.state_dict())
# The teammate's notebook, as they ran it: an earlier cell had started a quick fine-tuning check.
net.train()                       # left over from that cell
for bs in (400, 8, 1):
    print(f'teammate_evaluate, batch size {bs:3d}: mIoU {teammate_evaluate(net, DATA["val_standard"], DATA["val_mask"], bs):.3f}')
net.eval()
print(f'Afterwards, the correct evaluation of the same model object: '
      f'mIoU {segmentation_scores(predict_masks(net, DATA["val_standard"]), DATA["val_mask"])["miou"]:.3f}')
_ = net.load_state_dict(CLEAN_STATE)   # restore the trained weights and statistics

In [ ]:
# TODO T4a: the diagnostic experiment. Fill in the three marked lines; keep the rest.
rows = []
for mode in ('train', 'eval'):
    for bs in (1, 8, 400):
        _ = net.load_state_dict(CLEAN_STATE)
        # TODO: put `net` in the given mode
        # TODO: predict the standard view with this batch size (as teammate_evaluate does) and score it
        # TODO: record whether the BatchNorm running statistics changed
        raise NotImplementedError('T4a: complete the diagnostic loop.')
        rows.append(dict(mode=mode, batch_size=bs, miou=miou, running_stats_changed=changed))
_ = net.load_state_dict(CLEAN_STATE)
net.eval()
display(pd.DataFrame(rows).round(3))

In [ ]:
#@title A strange extra finding: the same bug on the harsh view
for bs in (400, 1):
    _ = net.load_state_dict(CLEAN_STATE)
    net.train()
    print(f'harsh view, teammate_evaluate (batch {bs:3d}): mIoU {teammate_evaluate(net, DATA["val_harsh"], DATA["val_mask"], bs):.3f}')
_ = net.load_state_dict(CLEAN_STATE)
net.eval()
print(f'harsh view, correct evaluation:            mIoU {segmentation_scores(predict_masks(net, DATA["val_harsh"]), DATA["val_mask"])["miou"]:.3f}')

### T4a · Your answer

**The cause, using your table (which rows differ, and why does batch size matter only in one mode?):** YOUR ANSWER
**Why did the model "get worse" after the teammate's evaluation, and what is the one-line fix?** YOUR ANSWER
**On the harsh view the buggy evaluation scores *higher* with a batch of 400. Why might that happen, and why is it still not a valid evaluation of the deployed model?** YOUR ANSWER (2–3 sentences)

### T4b · Plan (write before coding)

**Where exactly is the shortcut added, and why is the last ReLU after the addition rather than inside the branch?** YOUR ANSWER
**When does the shortcut need a 1×1 convolution, and what goes wrong without it?** YOUR ANSWER

In [ ]:
class ResidualBlock(nn.Module):
    """Slide 52: x -> conv-BN-ReLU -> conv-BN -> (+ shortcut(x)) -> ReLU."""
    def __init__(self, c_in, c_out):
        super().__init__()
        self.conv1, self.bn1 = nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out)
        self.conv2, self.bn2 = nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out)
        self.shortcut = None   # TODO T4b: identity when the shapes already match, otherwise 1x1 conv + BN

    def forward(self, x):
        # TODO T4b
        raise NotImplementedError('T4b: write the residual block.')

In [ ]:
#@title Checks for T4b, then the gradient diagnostic
for c_in, c_out in ((16, 16), (8, 16)):
    block = ResidualBlock(c_in, c_out).eval()
    x = torch.randn(2, c_in, 12, 12)
    assert block(x).shape == (2, c_out, 12, 12), 'Wrong output shape.'
    with torch.no_grad():
        block.bn2.weight.zero_(); block.bn2.bias.zero_()          # switch the residual branch off
        assert torch.allclose(block(x), F.relu(block.shortcut(x)), atol=1e-6), \
            'With the branch switched off, the block must return ReLU(shortcut(x)).'
assert isinstance(ResidualBlock(16, 16).shortcut, nn.Identity), 'Equal channels need no projection.'
print('T4b checks passed.')

def deep_net(depth, kind, w=16):
    """Stride-2 stem (24 -> 12), `depth` 3x3 conv layers, 1x1 classifier, upsample back to 24 x 24."""
    if kind == 'plain':
        body = [m for _ in range(depth) for m in (nn.Conv2d(w, w, 3, padding=1), nn.ReLU())]
    elif kind == 'bn':
        body = [m for _ in range(depth) for m in (*conv_bn_relu(w, w),)]
    else:
        body = [ResidualBlock(w, w) for _ in range(depth // 2)]     # two conv layers per block
    return nn.Sequential(nn.Conv2d(1, w, 3, stride=2, padding=1), nn.ReLU(), *body, nn.Conv2d(w, NC, 1),
                         nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False))

def gradient_table(depths=(4, 8, 16, 32)):
    rows = []
    x, y = normalise(DATA['train_img'][:64]), DATA['train_mask'][:64]
    for depth in depths:
        for kind in ('plain', 'bn', 'residual'):
            set_seed(0)
            model = deep_net(depth, kind).train()
            F.cross_entropy(model(x), y).backward()
            convs = [m for m in model.modules() if isinstance(m, nn.Conv2d) and m.kernel_size == (3, 3)]
            first, last = convs[0].weight.grad.norm().item(), convs[-1].weight.grad.norm().item()
            rows.append(dict(depth=depth, kind=kind, first_layer_grad=first, last_layer_grad=last, ratio=first / last))
    return pd.DataFrame(rows)

In [ ]:
# TODO T4b: predict BEFORE running: the order of magnitude (a power of ten, e.g. 1e-1 or 1e-9)
#           of first_layer_grad / last_layer_grad for a 32-layer network of each kind.
T4B_PREDICTION = dict(plain_32=None, bn_32=None, residual_32=None)

assert all(isinstance(v, (int, float)) for v in T4B_PREDICTION.values()), 'Write the three predictions first.'
grads = gradient_table()
display(grads.pivot(index='depth', columns='kind', values='ratio').map(lambda v: f'{v:.1e}'))
print('Your predictions for depth 32:', {k: f'{v:.0e}' for k, v in T4B_PREDICTION.items()})

In [ ]:
# TODO T4c: predict the standard-view mIoU of each 16-layer network BEFORE the runs.
predict('T4', {
    'T4_deep_plain':    dict(standard_miou=None),
    'T4_deep_bn':       dict(standard_miou=None),
    'T4_deep_residual': dict(standard_miou=None),
})

In [ ]:
def deep_plain(): return deep_net(16, 'plain')
def deep_bn(): return deep_net(16, 'bn')
def deep_residual(): return deep_net(16, 'residual')

t4_runs = [run_experiment(f'T4_{fn.__name__}', fn, 'T4') for fn in (deep_plain, deep_bn, deep_residual)]
display(experiment_table(['T4']).drop(columns=['task']))
display(prediction_check('T4').round(3))
plot_losses(t4_runs, 'T4c: 16 layers, same budget')

### T4b–c · Your answer

**Gradient table: how did the ratio change with depth for each kind, and how close were your predictions?** YOUR VALUES AND ANSWER
**Training results: rank the three networks, link the worst one's mIoU to T1a, and explain the ranking using the gradient table and the loss curves:** YOUR ANSWER (3–4 sentences)

## T5 · Augmentation you can trust, then test once · 4 points · 12 minutes

**a. Audit before you trust.** An AI assistant suggested `ai_augment` below ("random zoom, flip and brightness for segmentation"). It runs without errors. Write `audit_augmentation` with **three automatic checks**, then run it on three functions: identity (must pass everything), `ai_augment`, and `paired_flip` (flips image and mask together). For each check, write which bug it catches. Then name the problem that *no* automatic check here can catch. **(2 points, including the plan)**

**b. Test a claim (at most two runs, task `T5`).** Use the engine's photometric augmentation (`aug=` a condition dict like `HARSH`, applied to half of each batch; masks stay valid by construction). Choose one path and record predictions first. **(1 point, including the declaration at the top)**

- **Path A — you used an AI assistant:** ask it for one augmentation setting (illumination, offset, clutter or noise ranges) that should improve harsh-view mIoU. Test it.
- **Path B — supplied claim:** *"Pencil clutter, not lighting, is what breaks the model on harsh images. Clutter-only augmentation will recover most of the harsh-view mIoU."* Use your two runs to test it.

**c. Freeze, then test once.** The provided rule selects the T3, T4 or T5 run with the highest **harsh-view validation mIoU** (ties: fewer parameters). Write the decision first, then open the test set. **(1 point)**

In [ ]:
#@title The AI-suggested augmentation (provided; read it carefully)
def ai_augment(image, mask, gen):
    """'Random zoom, flip and brightness for segmentation' (suggested by an AI assistant)."""
    n = image.shape[0]
    size = int(round(CANVAS * (1 + 0.25 * torch.rand(1, generator=gen).item())))     # zoom in up to 25 %
    image = F.interpolate(image, size=size, mode='bilinear', align_corners=False)
    mask = F.interpolate(mask[:, None].float(), size=size, mode='bilinear', align_corners=False)[:, 0].long()
    top, left = (int(torch.randint(0, size - CANVAS + 1, (1,), generator=gen)) for _ in range(2))
    image, mask = image[..., top:top + CANVAS, left:left + CANVAS], mask[..., top:top + CANVAS, left:left + CANVAS]
    if torch.rand(1, generator=gen) < 0.5:
        image = image.flip(-1)
    if torch.rand(1, generator=gen) < 0.5:
        mask = mask.flip(-1)
    image = (image * (0.7 + 0.6 * torch.rand(n, 1, 1, 1, generator=gen))).clamp(0, 1)
    return image, mask

def identity_augment(image, mask, gen):
    return image, mask

def paired_flip(image, mask, gen):
    """Mirror image and mask together (the same random choice for both)."""
    flip = torch.rand(image.shape[0], generator=gen) < 0.5
    return (torch.where(flip[:, None, None, None], image.flip(-1), image),
            torch.where(flip[:, None, None], mask.flip(-1), mask))

def ink_label_agreement(image, mask, threshold=0.2):
    """Per-scene IoU between 'bright' pixels of a clean ink image and labelled digit pixels."""
    ink, lab = image[:, 0] > threshold, mask > 0
    return (ink & lab).flatten(1).sum(1).float() / (ink | lab).flatten(1).sum(1).clamp(min=1).float()

### T5a · Plan (write before coding)

**Read `ai_augment` line by line. Which lines do you suspect, and what would each do to the labels?** YOUR ANSWER
**Your three checks, in one line each (what is measured, and the pass condition):** YOUR ANSWER

In [ ]:
def audit_augmentation(fn, trials=20):
    """Apply `fn` to clean validation scenes many times; return {check_name: passed}."""
    gen = torch.Generator().manual_seed(0)
    ink, mask = DATA['val_ink'][:64], DATA['val_mask'][:64]
    passed = {'labels_valid': True, 'image_mask_aligned': True, 'no_new_classes': True}
    for _ in range(trials):
        img_a, mask_a = fn(ink.clone(), mask.clone(), gen)
        # TODO T5a check 1 'labels_valid': integer dtype, same shape as the input mask, values in 0..NC-1.
        # TODO T5a check 2 'image_mask_aligned': ink and labels must still coincide in every scene.
        #      Use ink_label_agreement(img_a, mask_a); choose the pass threshold by looking at identity_augment.
        # TODO T5a check 3 'no_new_classes': no scene may contain a label that was not in that scene before.
        raise NotImplementedError('T5a: write the three checks.')
    return passed

In [ ]:
#@title Run your audit
audit = pd.DataFrame({fn.__name__: audit_augmentation(fn) for fn in (identity_augment, ai_augment, paired_flip)}).T
display(audit)
assert audit.loc['identity_augment'].all(), 'Identity must pass every check: loosen a threshold or fix a check.'
assert not audit.loc['ai_augment'].all(), 'Your checks should catch at least one problem in ai_augment.'
gen = torch.Generator().manual_seed(1)
img_a, mask_a = ai_augment(DATA['val_ink'][:6].clone(), DATA['val_mask'][:6].clone(), gen)
show_masks([DATA['val_ink'][:6], DATA['val_mask'][:6], img_a, mask_a],
           ['clean ink', 'mask', 'ai_augment image', 'ai_augment mask'], n=6)

### T5a · Your answer

**Which of your checks failed for `ai_augment`, and which line of its code causes each failure?** YOUR ANSWER
**`paired_flip` passes every automatic check. Should you use it for digits? What can no automatic check here catch?** YOUR ANSWER (2 sentences)

### T5b · Path, claim and prediction (before running)

**Path (A or B), and the setting(s) you will test:** YOUR ANSWER
**Predicted harsh-view and standard-view mIoU compared with your T3 U-Net, and what result would contradict the claim:** YOUR ANSWER

In [ ]:
# TODO T5b: one or two augmentation settings. Each is a condition dict with the keys of STANDARD/HARSH:
#   amp (lighting strength range), offset (brightness-ramp range), strokes (clutter count range), noise (std).
CLAIM_PATH = None          # 'A' or 'B'
T5_SETTINGS = {            # run name -> condition dict
    # 'T5_clutter_only': dict(amp=(0.0, 0.3), offset=(0.0, 0.1), strokes=(3, 6), noise=0.05),
}

assert CLAIM_PATH in ('A', 'B') and 1 <= len(T5_SETTINGS) <= 2, 'Choose a path and one or two settings.'
for cond in T5_SETTINGS.values():
    assert set(cond) == set(STANDARD), 'Use exactly the keys amp, offset, strokes, noise.'
# TODO T5b: record predictions for every T5 run (and, if you like, a reference value for T3_unet).
predict('T5', {name: dict(harsh_miou=None, standard_miou=None) for name in T5_SETTINGS})

In [ ]:
t5_runs = [run_experiment(name, unet, 'T5', aug=cond) for name, cond in T5_SETTINGS.items()]
display(experiment_table(['T3', 'T5']).query("model_fn == 'unet'").drop(columns=['task']))
display(prediction_check('T5').round(3))
show_predictions([RUNS['T3_unet']] + t5_runs, view='harsh')

### T5b · Your answer

**Results versus the T3 U-Net on both views, and versus your predictions:** YOUR VALUES
**Does the evidence support the claim (or your AI's suggestion)? Use foreground IoU and digit accuracy to say *which* failure the augmentation fixed:** YOUR ANSWER (3 sentences)

In [ ]:
candidates = experiment_table(['T3', 'T4', 'T5']).query('standard_miou == standard_miou')     # completed runs
display(candidates.sort_values(['harsh_miou', 'parameters'], ascending=[False, True])[
    ['name', 'parameters', 'standard_miou', 'harsh_miou', 'harsh_fg_iou', 'harsh_digit_acc']])
rule_choice = candidates.sort_values(['harsh_miou', 'parameters'], ascending=[False, True]).iloc[0]['name']
final_run = RUNS[rule_choice]
print(f"Rule choice: {rule_choice} | harsh validation mIoU {final_run['harsh_miou']:.3f} | "
      f"standard {final_run['standard_miou']:.3f}")

### T5c · Decision before opening the test

**Final run, its validation evidence, and one limitation of choosing it this way:** YOUR ANSWER (3 sentences)

In [ ]:
DECISION_WRITTEN = False   # TODO T5c: set True only after writing the decision above.
assert DECISION_WRITTEN, 'Write and freeze the decision before opening the test.'
if 'final_report' not in globals():
    final_report = open_test_once(final_run)
for view in ('standard', 'harsh'):
    s = final_report[view]
    print(f"test {view:8s}: mIoU {s['miou']:.3f} | foreground IoU {s['fg_iou']:.3f} | digit accuracy {s['digit_acc']:.3f}"
          f" | validation mIoU {final_run[view + '_miou']:.3f}")
show_masks([final_report['images'][:6], final_report['mask'][:6], final_report['harsh_pred'][:6]],
           ['test (harsh)', 'truth', final_report['run_name']], n=6)

### T5c · Final interpretation

**Test results on both views, compared cautiously with validation:** YOUR VALUES AND ANSWER (2 sentences)
**One remaining failure visible in the test pictures, and the next experiment you would run (proposal only):** YOUR ANSWER

In [ ]:
# Required final evidence: keep this table in the submitted notebook.
final_log = experiment_table()
display(final_log.round(3))
print('Recorded runs:', len(final_log), '| total training time:', round(final_log.seconds.sum(), 1), 's')
print('Predictions recorded for:', {k: v['time'] for k, v in PREDICTIONS.items()})
print('Final model:', final_report['run_name'], '| test opened once:', TEST_OPENED)

## Check and submit · final 4 minutes

1. Fill in your name, Neptun code and assistance declaration. Replace every `YOUR ANSWER`/`YOUR VALUES` and complete all TODOs, including the plan boxes.
2. Keep all outputs visible: prediction tables, run tables, pictures, the gradient table, the audit, the final log. Unsuccessful runs are evidence; do not delete them.
3. Check the notebook from top to bottom. If you restart and rerun, reproduce the **same frozen plan** and the same predictions; do not change them after seeing results.
4. Rename the notebook **`YOUR_NEPTUN_DL_Lab05.ipynb`**, replacing `YOUR_NEPTUN` with your Neptun code, then choose **File → Download → Download .ipynb**.
5. Upload the executed notebook to **Moodle by the end of the lab** and confirm the submission. A Colab link alone is not sufficient.

If you cannot finish, submit your partial work on time and state what is missing. If you do not have Moodle access, keep the downloaded file, tell the instructor before the deadline and email the work as described in the course README. Keep completed answers and personal details out of the public repository.

## Deeper investigations · optional, after the core

Use `task='extension'` for any training. **Results here cannot change your T5c decision.** For each investigation, write a prediction first and a 2–3 sentence conclusion.

- **D1 · Self-supervised pre-training (slides 43–46).** Pre-train the U-Net encoder on the *pretext task* of predicting how far a scene was rotated (0°, 90°, 180°, 270°), using training images without masks. Then fine-tune on only 10 % of the labelled scenes and compare with training from scratch on the same 10 %. Does the pretext task help, and why might rotation be a weak pretext for digits?
- **D2 · Seeds.** Retrain your final configuration and `T3_unet` with seeds 7 and 17 (`seed=`). Is the T5c ranking larger than the seed-to-seed spread?
- **D3 · Segmentation as clustering (slide 73).** Cluster the pixels of each harsh image into two groups by brightness (k-means) and treat the brighter group as foreground. Compare its foreground IoU with your best network. What does clustering lack?
- **D4 · Upsampling.** Replace bilinear upsampling in `TinyUNet` with `nn.ConvTranspose2d(c, c, 2, stride=2)`. Compare parameters and mIoU, and look for checkerboard patterns in the predictions.

In [ ]:
# Optional: your deeper-investigation code.

### Deeper investigations · your notes (optional)

YOUR ANSWER

## References and credits

- BME lecture: *Deep Learning — Computer Vision* (27 October 2026), supplied course slides: self-supervised learning, BatchNorm and ResNets, data annotation and augmentation, semantic segmentation.
- [scikit-learn: the 8×8 digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), derived from the UCI Optical Recognition of Handwritten Digits data (E. Alpaydin, C. Kaynak). This is **not MNIST**. The 24×24 scenes, lighting and clutter are artificial classroom transformations, not a benchmark.
- Long, Shelhamer & Darrell (2015), [Fully Convolutional Networks for Semantic Segmentation](https://arxiv.org/abs/1411.4038); Ronneberger, Fischer & Brox (2015), [U-Net](https://arxiv.org/abs/1505.04597).
- Ioffe & Szegedy (2015), [Batch Normalization](https://arxiv.org/abs/1502.03167); He et al. (2016), [Deep Residual Learning](https://arxiv.org/abs/1512.03385).
- Gidaris, Singh & Komodakis (2018), [Unsupervised Representation Learning by Predicting Image Rotations](https://arxiv.org/abs/1803.07728).
- PyTorch documentation: [`nn.BatchNorm2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html), [`F.interpolate`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html), [`F.unfold`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.unfold.html).

Course material: Dr. Mohammed Salah Al-Radhi, BME. Dataset and framework credits remain with their authors.